In [3]:
from dotenv import load_dotenv
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers.audio import OpenAIWhisperParser
from langchain_community.document_loaders import YoutubeAudioLoader

from langchain_community.document_loaders import YoutubeLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS

In [4]:
loader = YoutubeLoader.from_youtube_url(
    "https://www.youtube.com/watch?v=dQw4w9WgXcQ",
    add_video_info=False  # Set True to include title, author, etc.
)

docs = loader.load()
print(docs[0].page_content)  # The transcript text

[♪♪♪] ♪ We're no strangers to love ♪ ♪ You know the rules
and so do I ♪ ♪ A full commitment's
what I'm thinking of ♪ ♪ You wouldn't get this
from any other guy ♪ ♪ I just wanna tell you
how I'm feeling ♪ ♪ Gotta make you understand ♪ ♪ Never gonna give you up ♪ ♪ Never gonna let you down ♪ ♪ Never gonna run around
and desert you ♪ ♪ Never gonna make you cry ♪ ♪ Never gonna say goodbye ♪ ♪ Never gonna tell a lie
and hurt you ♪ ♪ We've known each other
for so long ♪ ♪ Your heart's been aching
but you're too shy to say it ♪ ♪ Inside we both know
what's been going ♪ ♪ We know the game
and we're gonna play it ♪ ♪ And if you ask me
how I'm feeling ♪ ♪ Don't tell me
you're too blind to see ♪ ♪ Never gonna give you up ♪ ♪ Never gonna let you down ♪ ♪ Never gonna run around
and desert you ♪ ♪ Never gonna make you cry ♪ ♪ Never gonna say goodbye ♪ ♪ Never gonna tell a lie
and hurt you ♪ ♪ Never gonna give you up ♪ ♪ Never gonna let you down ♪ ♪ Never gonna run around
and desert you ♪ ♪ Never gon

In [5]:
!uv add youtube_transcript_api

Resolved 358 packages in 4ms
Audited 332 packages in 0.47ms


In [5]:
loader = YoutubeLoader.from_youtube_url(
    "https://www.youtube.com/watch?v=dQw4w9WgXcQ",
    add_video_info=False,       # Fetches title, author, views, etc.
    language=["en", "fr"],     # Preferred transcript language       # Auto-translate if needed
)

docs = loader.load()
print(docs[0].metadata)
# {'source': 'dQw4w9WgXcQ', 'title': '...', 'author': '...', 'view_count': ...}

{'source': 'dQw4w9WgXcQ'}


In [6]:
print(docs[0].page_content)  # The transcript text

[♪♪♪] ♪ We're no strangers to love ♪ ♪ You know the rules
and so do I ♪ ♪ A full commitment's
what I'm thinking of ♪ ♪ You wouldn't get this
from any other guy ♪ ♪ I just wanna tell you
how I'm feeling ♪ ♪ Gotta make you understand ♪ ♪ Never gonna give you up ♪ ♪ Never gonna let you down ♪ ♪ Never gonna run around
and desert you ♪ ♪ Never gonna make you cry ♪ ♪ Never gonna say goodbye ♪ ♪ Never gonna tell a lie
and hurt you ♪ ♪ We've known each other
for so long ♪ ♪ Your heart's been aching
but you're too shy to say it ♪ ♪ Inside we both know
what's been going ♪ ♪ We know the game
and we're gonna play it ♪ ♪ And if you ask me
how I'm feeling ♪ ♪ Don't tell me
you're too blind to see ♪ ♪ Never gonna give you up ♪ ♪ Never gonna let you down ♪ ♪ Never gonna run around
and desert you ♪ ♪ Never gonna make you cry ♪ ♪ Never gonna say goodbye ♪ ♪ Never gonna tell a lie
and hurt you ♪ ♪ Never gonna give you up ♪ ♪ Never gonna let you down ♪ ♪ Never gonna run around
and desert you ♪ ♪ Never gon

In [10]:
import os 

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")

In [11]:
from langchain_community.document_loaders import YoutubeLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter  # moved here too
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. Load
loader = YoutubeLoader.from_youtube_url(
    "https://www.youtube.com/watch?v=dQw4w9WgXcQ",
    add_video_info=False,
    language=["en", "en-US"],
)
docs = loader.load()

# 2. Split
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(docs)

# 3. Embed & Store
vectorstore = FAISS.from_documents(chunks, OpenAIEmbeddings(model="text-embedding-3-small"))
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# 4. Build chain using pure LCEL (no langchain.chains at all)
prompt = ChatPromptTemplate.from_template("""
Answer the question using ONLY the context below.
If the answer isn't there, say "Not covered in the video."

Context: {context}

Question: {question}
""")

llm = ChatOpenAI(model="gpt-4o", temperature=0)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# LCEL pipe: retrieve → format → prompt → llm → parse
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 5. Ask
answer = chain.invoke("What is this video about?")
print(answer)

Not covered in the video.


In [12]:
urls = ["https://www.youtube.com/watch?v=dQw4w9WgXcQ"]
save_dir = "./youtube_audio"

loader = GenericLoader(
    YoutubeAudioLoader(urls, save_dir),
    OpenAIWhisperParser()
)

docs = loader.load()
print(docs[0].page_content)

[youtube] Extracting URL: https://www.youtube.com/watch?v=dQw4w9WgXcQ
[youtube] dQw4w9WgXcQ: Downloading webpage


[youtube] dQw4w9WgXcQ: Downloading android vr player API JSON
[info] dQw4w9WgXcQ: Downloading 1 format(s): 140
[download] youtube_audio\Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster).m4a has already been downloaded
[download] 100% of    3.29MiB
[ExtractAudio] Not converting audio youtube_audio\Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster).m4a; file is already in target format m4a
Transcribing part 1!
Attempt 1 failed. Exception: Connection error.
Attempt 2 failed. Exception: Connection error.
We're no strangers to love You know the rules and so do I I've full commitments while I'm thinking of You wouldn't get this from any other guy I just wanna tell you how I'm feeling Gotta make you understand Never gonna give you up Never gonna let you down Never gonna run around and desert you Never gonna make you cry Never gonna say goodbye Never gonna tell a lie and hurt you We've known each other for so long Your heart's been aching but you're too shy

In [13]:
from langchain_community.document_loaders import FileSystemBlobLoader

In [17]:
url="https://www.youtube.com/watch?v=jGwO_UgTS7I"
save_dir="./youtube_audio/"
loader = GenericLoader(
    #YoutubeAudioLoader([url],save_dir),  # fetch from youtube
    FileSystemBlobLoader(save_dir, glob="*.m4a"),   #fetch locally
    OpenAIWhisperParser()
)
docs = loader.load()

Transcribing part 1!
Attempt 1 failed. Exception: Connection error.
Attempt 2 failed. Exception: Connection error.


In [18]:
docs

[Document(metadata={'source': 'youtube_audio\\Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster).m4a', 'chunk': 0}, page_content="We're no strangers to love You know the rules and so do I I've full commitments while I'm thinking of You wouldn't get this from any other guy I just wanna tell you how I'm feeling Gotta make you understand Never gonna give you up Never gonna let you down Never gonna run around and desert you Never gonna make you cry Never gonna say goodbye Never gonna tell a lie and hurt you We've known each other for so long Your heart's been aching but you're too shy to say it Inside we both know what's been going on We know the game and we're gonna play it And if you ask me how I'm feeling Don't tell me you're too blind to see Never gonna give you up Never gonna let you down Never gonna run around and desert you Never gonna make you cry Never gonna say goodbye Never gonna tell a lie and hurt you Never gonna give you up Never gonna let you down Never gon

In [19]:
docs[0].page_content[0:500]

"We're no strangers to love You know the rules and so do I I've full commitments while I'm thinking of You wouldn't get this from any other guy I just wanna tell you how I'm feeling Gotta make you understand Never gonna give you up Never gonna let you down Never gonna run around and desert you Never gonna make you cry Never gonna say goodbye Never gonna tell a lie and hurt you We've known each other for so long Your heart's been aching but you're too shy to say it Inside we both know what's been "